# MDLF Extended — Individual AP + Neighborhood Crowd activados

Igual que `prueba_MDLF.ipynb` pero con las dos features opcionales activas:
- `include_prox=True` → D_ind = 8 (añade sub-features de punto de acceso próximo)
- `include_neighborhood_crowd=True` → D_nbcrowd = 2 (crowd filtrado al vecindario de residencia)
- `line_ap_path` → **embedding LINE óptimo de Optuna** (`line_cercanias_optuna.parquet`, Spearman r=0.917)

**Dimensiones resultantes:**
- `D_static = D_ind(8) + D_crowd(2) + D_nbcrowd(2) = 12`
- `V_static ∈ R^(n, N, 12)` — vs 8 en la versión base
- `V ∈ R^(n, N, 16)` sumando H=4 (OD Region Correlation)

## Ejecución principal

In [ ]:
import os, sys
from pathlib import Path

# Forzar recarga de pipeline_mdlf para evitar módulo cacheado con versión antigua
for _k in list(sys.modules.keys()):
    if 'pipeline_mdlf' in _k or ('MDLF' in _k and 'pipeline' in _k):
        del sys.modules[_k]

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()

if not (_cubo_dir / 'pipeline.py').exists():
    raise FileNotFoundError(f"pipeline.py no encontrado en {_cubo_dir}")

os.chdir(_cubo_dir)
if str(_cubo_dir) not in sys.path:
    sys.path.insert(0, str(_cubo_dir))

print(f"CWD → {os.getcwd()}")

from pipeline import *

# ── Hiperparámetros del pipeline principal ────────────────────────────────────
hip = {
    'radio_busqueda': 680,
    'radio_busqueda_joven': 615,
    'radio_busqueda_adulto': 680,
    'radio_busqueda_3a_edad': 580,
    'radio_busqueda_turista': 680,
    'radio_busqueda_tarjeta_azul': 680,
    'radio_busqueda_infantil': 680,
    'tipo_validacion': 'TODO',
    'perfil_unico': 'no',
    'condicion_caso_ajustado': 300,
    'paper_original': 'munizaga',
    'dia_virtual': True,
}

# ── Hiperparámetros del módulo MDLF ──────────────────────────────────────────
_line_optuna = _cubo_dir.parent / 'MDLF' / 'LINE' / 'output' / 'line_cercanias_optuna.parquet'

mdlf_params = {
    # --- Matrices OD para LDA ---
    'od_matrices_subdirs': [
        'validaciones_3_7_febrero',
        'validaciones_8_9_febrero',
        'validaciones_10_16_febrero',
        'validaciones_17_23_febrero',
        'validaciones_24febrero_2marzo',
    ],

    # --- Historial individual (enero + febrero → señal suficiente para residencia e individual) ──
    'hist_individual_paths': [
        str(_cubo_dir / 'historial_enero_2025.csv'),
        str(_cubo_dir / 'historial_febrero_2025.csv'),
    ],

    # --- LINE (embedding óptimo de Optuna) ---
    'line_ap_path': str(_line_optuna),

    # --- LDA ---
    'n_topics_cercanias'    : 16,
    'lda_alpha'             : 0.5,
    'lda_beta'              : 0.01,
    'lda_max_iter'          : 200,
    'lda_filter_prefixes'   : ['04'],
    'lda_use_periods'       : True,
    'lda_min_df'            : 1,
    'lda_log_counts'        : False,
    'lda_token_granularity' : 'temporal',
    'lda_min_station_trips' : 0,
    'lda_n_init'            : 3,
    'lda_slot_minutes'      : 15,

    # --- Crowd Features ---
    'n_weeks': 2,

    # --- OD Region Correlation ---
    'n_heads': 4,
    'd_t': 16,

    # --- CNN ---
    'cnn_channels': 16,
    'kernel_size': 1,

    # --- Entrenamiento ---
    'optimizer': 'sgd',
    'lr': 0.01,
    'batch_size': 64,
    'n_epochs': 50,
    'lr_scheduler': True,

    # --- Features opcionales ---
    'include_prox': True,
    'include_neighborhood_crowd': True,
    'dominance_ratio': 2.0,
    'min_days_residence': 5,
    'min_neighbors': 10,

    # --- Otros ---
    'test_size': 0.2,
    'random_state': 42,
}

print(f"LINE path: {_line_optuna}")
print(f"  existe: {_line_optuna.exists()}")

## CREACION DE DATOS PARA HACER MDLF DIRECTAMENTE ##

In [ ]:
# Ejecuta el pipeline de cubo izquierdo con hipótesis 21 para guardar los 5 inputs
# en MDLF/almacen/50mil_tarjetas_3_al_5_marzo/
# Solo hay que hacer esto una vez; después se puede usar hipótesis 20 sin re-ejecutar cubo izquierdo.

pipe_almacen = Pipeline(
    str(_cubo_dir / '75mil_tarjetas_enero_junio_2025.csv'),
    write_results=False,
    hipotesis={5, 10, 21},   # 21 guarda los inputs, sin lanzar MDLF
    hiperparams=hip,
    mdlf_params={},
    name_out=None,
)
x_mal, y_mal, almacen = pipe_almacen.start()

_almacen_path = _cubo_dir.parent / 'MDLF' / 'almacen' / '75mil_tarjetas_enero_junio_2025'
print(f"\nInputs guardados en: {_almacen_path}")
print("Ficheros:")
for f in sorted(_almacen_path.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

## CELDA PRINCIPAL DE JUEGO ##

In [ ]:
import sys
from pathlib import Path

_estudios_dir = Path('/home/jupyter-lquijano/david/estudios')
if str(_estudios_dir) not in sys.path:
    sys.path.insert(0, str(_estudios_dir))

from MDLF.pipeline_mdlf import PipelineMDLF

_nombre_almacen = '50mil_tarjetas_3_al_5_marzo'
_line_optuna    = _estudios_dir / 'MDLF' / 'LINE' / 'output' / 'line_cercanias_optuna.parquet'
_ttp_paths      = [
    str(_estudios_dir / "descomprimido" / f"TTP_PERIODO_0{i}_2025.csv")
    for i in range(1, 7)
]

pipe_mdlf = PipelineMDLF(
    nombre_almacen=_nombre_almacen,
    ttp_paths=_ttp_paths,
    cubo_hiperparams=hip,
    od_matrices_subdirs=[
        'validaciones_3_7_febrero',
        'validaciones_8_9_febrero',
        'validaciones_10_16_febrero',
        'validaciones_17_23_febrero',
        'validaciones_24febrero_2marzo',
    ],
    line_ap_path=str(_line_optuna),
    # LDA
    n_topics_cercanias=16,
    lda_alpha=0.5,
    lda_beta=0.01,
    lda_max_iter=200,
    lda_filter_prefixes=['04'],
    lda_use_periods=True,
    lda_min_df=1,
    lda_log_counts=False,
    lda_token_granularity='temporal',
    lda_min_station_trips=0,
    lda_n_init=3,
    lda_slot_minutes=15,
    # Crowd
    n_weeks=2,
    # OD Region Correlation
    n_heads=4,
    d_t=16,
    # CNN
    cnn_channels=16,
    kernel_size=1,
    # Entrenamiento
    optimizer='adamw',
    lr=0.001,
    weight_decay=0.01,
    batch_size=64,
    n_epochs=50,
    lr_scheduler=True,
    # Features opcionales
    include_prox=True,
    include_neighborhood_crowd=True,
    dominance_ratio=2.0,
    min_days_residence=5,
    min_neighbors=10,
    # Otros
    test_size=0.2,
    random_state=42,
).run()

print(f"Top-1 accuracy MDLF: {pipe_mdlf.test_accuracy:.4f}")
print(f"Top-5 accuracy MDLF: {pipe_mdlf.test_accuracy5:.4f}")

## Estudio de Ablación

In [ ]:
# ── Parámetros base compartidos por todos los experimentos de ablación ─────────
# Igual que el run principal pero como dict reutilizable.
_base = dict(
    nombre_almacen=_nombre_almacen,
    ttp_paths=_ttp_paths,
    cubo_hiperparams=hip,
    od_matrices_subdirs=[
        'validaciones_3_7_febrero',
        'validaciones_8_9_febrero',
        'validaciones_10_16_febrero',
        'validaciones_17_23_febrero',
        'validaciones_24febrero_2marzo',
    ],
    line_ap_path=str(_line_optuna),
    n_topics_cercanias=16, lda_alpha=0.5, lda_beta=0.01, lda_max_iter=200,
    lda_filter_prefixes=['04'], lda_use_periods=True, lda_min_df=1, lda_log_counts=False,
    lda_token_granularity='temporal', lda_min_station_trips=0, lda_n_init=3, lda_slot_minutes=15,
    n_weeks=2, n_heads=4, d_t=16, cnn_channels=16, kernel_size=1,
    optimizer='adamw', lr=0.001, weight_decay=0.01, batch_size=64, n_epochs=50, lr_scheduler=True,
    include_prox=True, include_neighborhood_crowd=True,
    dominance_ratio=2.0, min_days_residence=5, min_neighbors=10,
    test_size=0.2, random_state=42,
)

# ── Ablación: sin Neighborhood Crowd (individual_ap activo) ────────────────────
pipe_no_nbcrowd = PipelineMDLF(**{**_base, 'include_neighborhood_crowd': False}).run()

print(f"[sin nbcrowd]  Top-1: {pipe_no_nbcrowd.test_accuracy:.4f} | Top-5: {pipe_no_nbcrowd.test_accuracy5:.4f}")

In [ ]:
# ── Ablación: sin Individual AP (neighborhood_crowd activo) ────────────────────
pipe_no_prox = PipelineMDLF(**{**_base, 'include_prox': False}).run()

print(f"[sin ind_ap]   Top-1: {pipe_no_prox.test_accuracy:.4f} | Top-5: {pipe_no_prox.test_accuracy5:.4f}")

In [ ]:
# ── Ablación: sin ninguna de las dos (equivalente a prueba_MDLF.ipynb base) ───
pipe_base = PipelineMDLF(**{**_base, 'include_prox': False, 'include_neighborhood_crowd': False}).run()

print(f"[base D=8]     Top-1: {pipe_base.test_accuracy:.4f} | Top-5: {pipe_base.test_accuracy5:.4f}")

In [ ]:
# ── Resumen comparativo ───────────────────────────────────────────────────────
results = {
    'Extended (ind_ap + nbcrowd)': pipe_mdlf,
    'Sin nbcrowd  (solo ind_ap)' : pipe_no_nbcrowd,
    'Sin ind_ap   (solo nbcrowd)': pipe_no_prox,
    'Base         (ninguna ext.)': pipe_base,
}

print(f"{'Configuración':<35} {'Top-1':>8} {'Top-5':>8}")
print("-" * 55)
for name, mdlf in results.items():
    print(f"{name:<35} {mdlf.test_accuracy:>8.4f} {mdlf.test_accuracy5:>8.4f}")

## AdamW vs SGD

In [ ]:
# ── SGD: misma config que el run principal pero con SGD en lugar de AdamW ─────
# AdamW gestiona mejor gradientes dispersos (E_table) y escalas heterogéneas
# entre bloques. lr ~10× mayor que AdamW es habitual con SGD.
pipe_sgd = PipelineMDLF(**{**_base, 'optimizer': 'sgd', 'lr': 0.01, 'weight_decay': 0.0}).run()

print(f"[AdamW]  Top-1: {pipe_mdlf.test_accuracy:.4f} | Top-5: {pipe_mdlf.test_accuracy5:.4f}")
print(f"[SGD]    Top-1: {pipe_sgd.test_accuracy:.4f}  | Top-5: {pipe_sgd.test_accuracy5:.4f}")

## Análisis de predicciones — Confusion Matrix y estadísticas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Obtener análisis del modelo principal ─────────────────────────────────────
mdlf = pipe.mdlf
analysis = mdlf.prediction_analysis(top_n=15)

cm_df        = analysis["confusion_matrix"]
per_station  = analysis["per_station"]
most_pred    = analysis["most_predicted"]
most_failed  = analysis["most_failed"]

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONFUSION MATRIX  (solo estaciones con ≥ 3 viajes reales en test)
# ═══════════════════════════════════════════════════════════════════════════════
min_real = 3
active = per_station.loc[per_station["n_real"] >= min_real, "estacion"].tolist()
cm_filt = cm_df.loc[active, active]

# Normalizar por filas (% de cada estación real que va a cada destino predicho)
cm_norm = cm_filt.div(cm_filt.sum(axis=1).replace(0, 1), axis=0) * 100

fig, ax = plt.subplots(figsize=(max(12, len(active) * 0.55), max(10, len(active) * 0.55)))
im = ax.imshow(cm_norm.values, aspect="auto", cmap="Blues", vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label="% de viajes reales predichos como destino (por fila)")

ax.set_xticks(range(len(active)))
ax.set_yticks(range(len(active)))
ax.set_xticklabels(active, rotation=90, fontsize=7)
ax.set_yticklabels(active, fontsize=7)
ax.set_xlabel("Destino PREDICHO", fontsize=10)
ax.set_ylabel("Destino REAL", fontsize=10)
ax.set_title(
    f"Confusion Matrix — MDLF (top-1)  |  Top-1 acc={mdlf.test_accuracy:.1%}\n"
    f"(solo estaciones con ≥{min_real} viajes reales en test, normalizada por fila)",
    fontsize=11,
)

# Anotar celdas de la diagonal (aciertos)
for i in range(len(active)):
    val = cm_norm.values[i, i]
    if val > 0:
        color = "white" if val > 60 else "black"
        ax.text(i, i, f"{val:.0f}%", ha="center", va="center", fontsize=6, color=color, fontweight="bold")

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 2. ESTACIONES MÁS PREDICHAS vs ESTACIONES MÁS REALES
# ═══════════════════════════════════════════════════════════════════════════════
top_n = 15
top_pred  = most_pred.head(top_n)
top_real  = per_station.set_index("estacion")["n_real"].sort_values(ascending=False).head(top_n)
top_real_pct = (top_real / top_real.sum() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Más predichas
colors_pred = [
    "#d62728" if est not in top_real.index else "#1f77b4"
    for est in top_pred.index
]
axes[0].barh(top_pred.index[::-1], top_pred.values[::-1], color=colors_pred[::-1])
axes[0].set_xlabel("% del total de predicciones")
axes[0].set_title(f"Top-{top_n} estaciones más predichas\n(rojo = no está entre las más frecuentes en test real)")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter())
for i, (val, est) in enumerate(zip(top_pred.values[::-1], top_pred.index[::-1])):
    acc = per_station.loc[per_station["estacion"] == est, "accuracy"]
    label = f" {val:.1f}%  (acc={float(acc.iloc[0]):.0%})" if len(acc) > 0 else f" {val:.1f}%"
    axes[0].text(val + 0.1, i, label, va="center", fontsize=8)
axes[0].set_xlim(0, top_pred.values.max() * 1.4)

# Más reales
axes[1].barh(top_real_pct.index[::-1], top_real_pct.values[::-1], color="#2ca02c")
axes[1].set_xlabel("% del total de viajes reales en test")
axes[1].set_title(f"Top-{top_n} estaciones más frecuentes como destino REAL en test")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
for i, val in enumerate(top_real_pct.values[::-1]):
    axes[1].text(val + 0.1, i, f" {val:.1f}%", va="center", fontsize=8)
axes[1].set_xlim(0, top_real_pct.values.max() * 1.4)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3. ACCURACY POR ESTACIÓN (para las más frecuentes en test)
# ═══════════════════════════════════════════════════════════════════════════════
top_est = per_station[per_station["n_real"] >= min_real].head(20)

fig, ax = plt.subplots(figsize=(14, 5))
colors_acc = ["#d62728" if a < 0.25 else "#ff7f0e" if a < 0.5 else "#2ca02c"
              for a in top_est["accuracy"]]
bars = ax.bar(range(len(top_est)), top_est["accuracy"] * 100, color=colors_acc)
ax.set_xticks(range(len(top_est)))
ax.set_xticklabels(top_est["estacion"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Top-1 accuracy (%)")
ax.set_title(f"Accuracy por estación (top-20 por volumen real, ≥{min_real} viajes)\n"
             f"verde ≥50%  |  naranja 25-50%  |  rojo <25%")
ax.axhline(mdlf.test_accuracy * 100, color="black", linestyle="--", linewidth=1.2,
           label=f"Media global ({mdlf.test_accuracy:.1%})")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for i, (acc, n) in enumerate(zip(top_est["accuracy"], top_est["n_real"])):
    ax.text(i, acc * 100 + 1, f"{acc:.0%}\n(n={n})", ha="center", va="bottom", fontsize=7)
ax.set_ylim(0, 115)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 4. PARES DE ERROR MÁS FRECUENTES
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
ax.barh(
    most_failed["par_real_predicho"][::-1],
    most_failed["pct_sobre_fallos"][::-1],
    color="#9467bd",
)
ax.set_xlabel("% sobre el total de fallos")
ax.set_title("Top-15 pares de confusión más frecuentes  (real → predicho incorrecto)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
for i, (val, n) in enumerate(zip(
    most_failed["pct_sobre_fallos"][::-1],
    most_failed["n_errores"][::-1],
)):
    ax.text(val + 0.05, i, f" {val:.1f}%  (n={n})", va="center", fontsize=8)
ax.set_xlim(0, most_failed["pct_sobre_fallos"].max() * 1.5)
plt.tight_layout()
plt.show()